In [1]:
# Step 1A: runtime inspection only. No MRI preprocessing or model execution.
import os, sys, json, shutil, platform, subprocess
from pathlib import Path
from datetime import datetime, timezone
root = Path('/content/FYP_MRI_LiYang')
root.mkdir(parents=True, exist_ok=True)
mem = {}
for line in Path('/proc/meminfo').read_text().splitlines():
    key, value = line.split(':', 1)
    mem[key] = int(value.strip().split()[0])
disk = shutil.disk_usage('/content')
report = {
    'record_id': 'sub-mindo304_ses-placebo',
    'checked_at_utc': datetime.now(timezone.utc).isoformat(),
    'python': sys.version.split()[0],
    'platform': platform.platform(),
    'logical_cpus': os.cpu_count(),
    'ram_total_gib': round(mem['MemTotal']/1024**2, 2),
    'ram_available_gib': round(mem['MemAvailable']/1024**2, 2),
    'disk_free_gib': round(disk.free/1024**3, 2),
    'tools_on_path': {name: shutil.which(name) for name in ['recon-all','fmriprep','N4BiasFieldCorrection','hd-bet']},
    'preprocessing_started': False,
    'qc_status': 'pending',
    'training_ready': False
}
if shutil.which('nvidia-smi'):
    gpu = subprocess.run(['nvidia-smi','--query-gpu=name,memory.total','--format=csv,noheader'], capture_output=True, text=True)
    report['gpu'] = gpu.stdout.strip() or gpu.stderr.strip()
else:
    report['gpu'] = 'No GPU assigned'
(root/'runtime_check.json').write_text(json.dumps(report, indent=2))
print(json.dumps(report, indent=2))
print('Saved runtime_check.json in', root)
print('Only resource inspection performed. Raw data and preprocessing are untouched.')


{
  "record_id": "sub-mindo304_ses-placebo",
  "checked_at_utc": "2026-11-09T14:22:53.976446+00:00",
  "python": "3.13.18",
  "platform": "Linux-6.6.159+-x86_64-with-glibc2.39",
  "logical_cpus": 2,
  "ram_total_gib": 12.67,
  "ram_available_gib": 11.7,
  "disk_free_gib": 107.62,
  "tools_on_path": {
    "recon-all": null,
    "fmriprep": null,
    "N4BiasFieldCorrection": null,
    "hd-bet": null
  },
  "preprocessing_started": false,
  "qc_status": "pending",
  "training_ready": false,
  "gpu": "No GPU assigned"
}
Saved runtime_check.json in /content/FYP_MRI_LiYang
Only resource inspection performed. Raw data and preprocessing are untouched.


In [2]:
# Step 1B: read pinned public metadata; no preprocessing.
import urllib.request, hashlib, csv, io
COMMIT = 'a911ab6b2120698fdf48f1936c7bf7c5c1a17a4a'
BASE = 'https://raw.githubusercontent.com/bykarl/MRI-classification/' + COMMIT + '/'
MANIFEST_SHA = '8c36e85113f3a5b5e853b471f664397c74ed79eca71e014cd198aa82076da0a2'
def get_bytes(url):
    request = urllib.request.Request(url, headers={'User-Agent':'FYP-MRI-input-check'})
    with urllib.request.urlopen(request, timeout=120) as response:
        return response.read()
manifest_bytes = get_bytes(BASE + 'dataset_Mi/dataset_manifest.json')
assert hashlib.sha256(manifest_bytes).hexdigest() == MANIFEST_SHA, 'Manifest hash mismatch'
manifest = json.loads(manifest_bytes)
pilot_bytes = get_bytes(BASE + 'manifests/pilot_subjects.csv')
pilot_rows = list(csv.DictReader(io.StringIO(pilot_bytes.decode('utf-8-sig'))))
record = next(r for r in pilot_rows if r['record_id']=='sub-mindo304_ses-placebo')
selected = [f for f in manifest['files'] if f['path'] in {record['t1_path'],record['bold_path']}]
assert len(selected)==2
(root/'pinned_manifest.json').write_bytes(manifest_bytes)
(root/'pilot_subjects.csv').write_bytes(pilot_bytes)
print(json.dumps({'source_commit':COMMIT,'manifest_sha256':MANIFEST_SHA,'record':record,'images':selected,'support_archive':manifest['support_archive']},indent=2))


{
  "source_commit": "a911ab6b2120698fdf48f1936c7bf7c5c1a17a4a",
  "manifest_sha256": "8c36e85113f3a5b5e853b471f664397c74ed79eca71e014cd198aa82076da0a2",
  "record": {
    "record_id": "sub-mindo304_ses-placebo",
    "participant_id": "sub-mindo304",
    "session_id": "ses-placebo",
    "person_id": "",
    "identity_status": "unconfirmed",
    "diagnosis": "CN",
    "label": "0",
    "age": "65",
    "sex": "F",
    "education_code": "12",
    "scanner_batch": "UHP_post_upgrade",
    "same_session": "True",
    "t1_path": "rawdata/sub-mindo304/ses-placebo/anat/sub-mindo304_ses-placebo_T1w.nii.gz",
    "bold_path": "rawdata/sub-mindo304/ses-placebo/func/sub-mindo304_ses-placebo_task-rest_bold.nii.gz",
    "t1_sha256": "89ea431ba0e912d1b73643a4ce691dbe13ad54da394b645942e4e12dceaf73b5",
    "bold_sha256": "7469f562c6fb134d53508297710e446a0d943ee4b196d2c06fb30cd8b1c2877d",
    "tr_sec": "2.0",
    "n_frames_source": "240",
    "diagnosis_source": "participants.tsv/group",
    "diagnosis_s

In [3]:
# Step 1C: download ONLY this record and two matching JSON sidecars.
# Verify bytes/SHA-256; existing mismatching files cause an error, never overwrite.
import zipfile, importlib.util
raw_root = root/'dataset_Mi'
cache = root/'download_cache'
cache.mkdir(exist_ok=True)
def file_sha(path):
    h = hashlib.sha256()
    with Path(path).open('rb') as stream:
        for block in iter(lambda: stream.read(1024*1024), b''):
            h.update(block)
    return h.hexdigest()
def fetch_verified(item, dest):
    dest = Path(dest)
    if dest.exists():
        assert dest.stat().st_size == item['bytes'] and file_sha(dest)==item['sha256'], 'Existing file mismatch: '+str(dest)
        return
    dest.parent.mkdir(parents=True,exist_ok=True)
    temp = dest.with_name(dest.name+'.partial')
    request = urllib.request.Request(item['url'], headers={'User-Agent':'FYP-MRI-input-check'})
    with urllib.request.urlopen(request, timeout=120) as response, temp.open('wb') as stream:
        shutil.copyfileobj(response, stream, 1024*1024)
    assert temp.stat().st_size == item['bytes'], 'File size mismatch'
    assert file_sha(temp)==item['sha256'], 'File hash mismatch'
    temp.replace(dest)
for item in selected:
    fetch_verified(item, raw_root/item['path'])
    print('VERIFIED MRI:', item['asset_name'], item['bytes'], 'bytes')
support = manifest['support_archive']
support_path = cache/support['asset_name']
fetch_verified(support,support_path)
sidecars = {}
with zipfile.ZipFile(support_path) as archive:
    for item in selected:
        relative = item['path'].removesuffix('.nii.gz')+'.json'
        matches = [name for name in archive.namelist() if name==relative or name.endswith('/'+relative)]
        assert len(matches)==1, 'Expected exactly one matching JSON: '+relative
        content = archive.read(matches[0])
        parsed = json.loads(content)
        dest = raw_root/relative
        if dest.exists():
            assert dest.read_bytes()==content, 'Existing JSON mismatch'
        else:
            dest.write_bytes(content)
        sidecars[item['path']] = parsed
        print('VERIFIED JSON:', dest.name, 'sha256='+file_sha(dest))
if importlib.util.find_spec('nibabel') is None:
    subprocess.check_call([sys.executable,'-m','pip','install','--quiet','nibabel'])
print('Raw files verified. No preprocessing performed. QC remains pending.')


VERIFIED MRI: sub-mindo304_ses-placebo_T1w.nii.gz 8425883 bytes
VERIFIED MRI: sub-mindo304_ses-placebo_task-rest_bold.nii.gz 60013116 bytes
VERIFIED JSON: sub-mindo304_ses-placebo_T1w.json sha256=8103294a55884ebdb6cc8ce2b6b1bf9776f933f78a40b84b5f80f719052aade3
VERIFIED JSON: sub-mindo304_ses-placebo_task-rest_bold.json sha256=3019b2d1ee26185db6306c6d9e543f44018b8faf39a4ffee45d7f1683de30182
Raw files verified. No preprocessing performed. QC remains pending.


In [4]:
# Step 1D: inspect all voxel values and geometry; do not modify NIfTI files.
import nibabel as nib, numpy as np
participants_bytes = get_bytes(BASE+'dataset_Mi/rawdata/participants.tsv')
participants = list(csv.DictReader(io.StringIO(participants_bytes.decode('utf-8-sig')),delimiter='\t'))
participant = next(r for r in participants if r['participant_id']==record['participant_id'])
checks = {
    'record_id':record['record_id'], 'source_commit':COMMIT,
    'checked_at_utc':datetime.now(timezone.utc).isoformat(),
    'manifest_sha256':MANIFEST_SHA,
    'software':{'python':sys.version.split()[0],'nibabel':nib.__version__,'numpy':np.__version__},
    'source_participant':participant,
    'images':{}, 'warnings':[],
    'file_integrity_status':'pass', 'preprocessing_started':False,
    'qc_status':'pending', 'training_ready':False,
    'identity_status':record['identity_status'],'diagnosis_status':record['diagnosis_status']
}
loaded = {}
for key, expected_ndim in [('t1_path',3),('bold_path',4)]:
    relative = record[key]
    path = raw_root/relative
    image = nib.load(path)
    data = image.get_fdata(dtype=np.float32)
    assert data.ndim==expected_ndim and np.isfinite(data).all(), 'Invalid dimensionality/voxel values'
    assert np.isfinite(image.affine).all() and abs(np.linalg.det(image.affine[:3,:3]))>1e-10, 'Invalid affine'
    assert np.std(data)>0, 'Constant image'
    json_path = path.with_name(path.name.removesuffix('.nii.gz')+'.json')
    metadata = sidecars[relative]
    units = image.header.get_xyzt_units()
    entry = {
        'path':relative,'bytes':path.stat().st_size,'sha256':file_sha(path),
        'shape':list(map(int,image.shape)),
        'zooms':list(map(float,image.header.get_zooms())),
        'orientation':list(nib.aff2axcodes(image.affine)),
        'units':list(units),'qform_code':int(image.header['qform_code']),
        'sform_code':int(image.header['sform_code']),
        'affine':image.affine.tolist(),
        'all_voxels_finite':True,'min':float(data.min()),'max':float(data.max()),
        'json_sha256':file_sha(json_path),
        'metadata_fields':{k:metadata.get(k) for k in ['RepetitionTime','SliceTiming','PhaseEncodingDirection','TotalReadoutTime','EchoTime','Manufacturer','MagneticFieldStrength','SeriesDescription','ProtocolName']}
    }
    if key=='bold_path':
        assert image.shape[3]==int(record['n_frames_source']), 'Unexpected timepoint count'
        tr = metadata.get('RepetitionTime')
        assert tr is not None and np.isclose(float(tr),float(record['tr_sec'])), 'JSON TR mismatch'
        entry['tr_sec_from_json']=float(tr)
        if units[1] in ['sec','msec','usec']:
            scale={'sec':1.,'msec':.001,'usec':.000001}[units[1]]
            assert np.isclose(image.header.get_zooms()[3]*scale,float(tr)), 'Header/JSON TR conflict'
        else:
            checks['warnings'].append('BOLD header time unit unspecified; JSON TR verified, header left unchanged.')
        entry['constant_frame_indices']=[i for i in range(data.shape[3]) if np.std(data[...,i])==0]
        assert not entry['constant_frame_indices'], 'Constant BOLD frame found'
        if not metadata.get('SliceTiming'):
            checks['warnings'].append('No SliceTiming metadata: do not assume slice-timing correction is available.')
        if not metadata.get('PhaseEncodingDirection') or metadata.get('TotalReadoutTime') is None:
            checks['warnings'].append('Distortion-correction metadata incomplete; assess separately before preprocessing.')
    elif units[1]=='unknown':
        checks['warnings'].append('T1 header time unit unspecified; not used for a static 3D T1. No header rewrite performed.')
    checks['images'][key]=entry
    loaded[key]=(image,data)
checks['warnings'].append('No motion estimation, brain extraction, registration or clinical/identity validation performed in Step 1.')
(root/'input_check.json').write_text(json.dumps(checks,indent=2))
print(json.dumps(checks,indent=2))
print('INPUT READABILITY AND INTEGRITY: PASS; preprocessing/QC/training readiness remain pending.')


{
  "record_id": "sub-mindo304_ses-placebo",
  "source_commit": "a911ab6b2120698fdf48f1936c7bf7c5c1a17a4a",
  "checked_at_utc": "2026-11-09T14:25:22.823938+00:00",
  "manifest_sha256": "8c36e85113f3a5b5e853b471f664397c74ed79eca71e014cd198aa82076da0a2",
  "software": {
    "python": "3.13.18",
    "nibabel": "5.4.2",
    "numpy": "2.4.6"
  },
  "source_participant": {
    "participant_id": "sub-mindo304",
    "age": "65",
    "sex": "F",
    "age_category": "older",
    "group": "CN",
    "education_years": "12",
    "mothers_education_years": "12",
    "race": "White",
    "ethnicity": "Not Hispanic or Latino",
    "hearing_aid": "no",
    "music_experience": "no",
    "music_years": "n/a",
    "moca": "23",
    "scanner_upgrade": "UHP_post_upgrade"
  },
  "images": {
    "t1_path": {
      "path": "rawdata/sub-mindo304/ses-placebo/anat/sub-mindo304_ses-placebo_T1w.nii.gz",
      "bytes": 8425883,
      "sha256": "89ea431ba0e912d1b73643a4ce691dbe13ad54da394b645942e4e12dceaf73b5",
     

In [5]:
# Step 1E: additional geometry audit. Never repair raw headers by guessing.
geometry = {}
for key,(image,data) in loaded.items():
    q,qcode = image.get_qform(coded=True)
    s,scode = image.get_sform(coded=True)
    header_spacing=np.array(image.header.get_zooms()[:3])
    affine_spacing=nib.affines.voxel_sizes(image.affine)
    same_spacing=bool(np.allclose(header_spacing,affine_spacing,rtol=1e-4,atol=1e-4))
    same_forms=bool(q is not None and s is not None and np.allclose(q,s,rtol=1e-4,atol=1e-3))
    geometry[key]={
        'header_pixdim_xyz':header_spacing.tolist(),
        'chosen_affine_spacing_xyz':affine_spacing.tolist(),
        'pixdim_matches_chosen_affine':same_spacing,
        'qform_code':int(qcode),'sform_code':int(scode),
        'qform':None if q is None else q.tolist(),
        'sform':None if s is None else s.tolist(),
        'qform_matches_sform':same_forms,
        'spatial_unit':image.header.get_xyzt_units()[0]
    }
    if not same_spacing:
        checks['warnings'].append(key+': header pixdim and chosen affine spacing conflict; resolve source geometry before spatial processing.')
    if not same_forms:
        checks['warnings'].append(key+': qform/sform differ; confirm intended coordinate mapping before registration.')
    if image.header.get_xyzt_units()[0]=='unknown':
        checks['warnings'].append(key+': spatial unit unspecified; require source evidence, not an implicit mm assumption.')
checks['geometry_audit']=geometry
checks['geometry_status']='review_required' if any(not g['pixdim_matches_chosen_affine'] or not g['qform_matches_sform'] or g['spatial_unit']=='unknown' for g in geometry.values()) else 'pass'
checks['ready_for_spatial_preprocessing']=checks['geometry_status']=='pass'
(root/'input_check_with_geometry.json').write_text(json.dumps(checks,indent=2))
print(json.dumps({'geometry':geometry,'geometry_status':checks['geometry_status'],'warnings':checks['warnings']},indent=2))
print('Raw MRI SHA-256 unchanged:',all(file_sha(raw_root/item['path'])==item['sha256'] for item in selected))


{
  "geometry": {
    "t1_path": {
      "header_pixdim_xyz": [
        1.0,
        1.0,
        1.0
      ],
      "chosen_affine_spacing_xyz": [
        0.9999999227864619,
        0.9999999759235806,
        1.0000000351137743
      ],
      "pixdim_matches_chosen_affine": true,
      "qform_code": 1,
      "sform_code": 2,
      "qform": [
        [
          -1.0,
          0.0,
          0.0,
          127.5
        ],
        [
          0.0,
          1.0,
          0.0,
          -95.625
        ],
        [
          0.0,
          0.0,
          1.0,
          -86.25
        ],
        [
          0.0,
          0.0,
          0.0,
          1.0
        ]
      ],
      "sform": [
        [
          -0.999332070350647,
          0.03649318218231201,
          0.0018731951713562012,
          124.34559631347656
        ],
        [
          0.036348819732666016,
          0.9875112175941467,
          0.15329807996749878,
          -148.0451202392578
        ],
        [
 

In [6]:
# Step 1F: raw display preview + portable reports. NOT registration/QC approval.
import matplotlib.pyplot as plt
preview_fig, axes=plt.subplots(2,3,figsize=(12,7))
for row,key in enumerate(['t1_path','bold_path']):
    image,data=loaded[key]
    volume=data if data.ndim==3 else data.mean(axis=3)
    display_img=nib.as_closest_canonical(nib.Nifti1Image(volume,image.affine))
    view=np.asarray(display_img.dataobj)
    positive=view[view>0]
    lo,hi=np.percentile(positive,[2,98])
    center=[n//2 for n in view.shape]
    planes=[view[center[0],:,:],view[:,center[1],:],view[:,:,center[2]]]
    for col,(plane,name) in enumerate(zip(planes,['Sagittal','Coronal','Axial'])):
        axes[row,col].imshow(plane.T,origin='lower',cmap='gray',vmin=lo,vmax=hi)
        axes[row,col].set_title(('T1' if row==0 else 'Mean raw BOLD')+' / '+name)
        axes[row,col].axis('off')
preview_fig.suptitle('sub-mindo304 / ses-placebo - RAW preview, no registration or QC approval')
preview_fig.tight_layout()
preview_fig.savefig(root/'raw_preview.png',dpi=140)
plt.show()
summary = '''STEP 1 - sub-mindo304_ses-placebo
Completed: runtime inspection; verified two MRI files and matching JSON; checked full voxel readability, dimensions, TR and source metadata.
T1: 256 x 256 x 156; 1 mm header spacing; LAS.
fMRI: 64 x 64 x 43 x 240; JSON TR 2 seconds.
Runtime: 2 logical CPUs, 12.67 GiB RAM, no GPU. FreeSurfer/fMRIPrep not installed.
Review required: BOLD header x-spacing 2.0 versus sform-derived 3.4375; BOLD space/time units unknown; qform/sform mapping differs; SliceTiming unavailable.
qform/sform differences alone do not prove corruption. Establish intended coordinate mapping from source evidence before harmonizing derived inputs.
No raw headers changed; no preprocessing, motion estimation, feature extraction or training performed.
Status: file integrity PASS; geometry REVIEW REQUIRED; QC PENDING; training_ready FALSE.
Raw files remain in the temporary Colab runtime; notebook outputs are saved in Drive. Rerun download cells if runtime is recycled.
Next: resolve source geometry and confirm the T1 input recipe before spatial preprocessing.
'''
(root/'STEP1_README.txt').write_text(summary)
checks['raw_preview_note']='Canonical axis reordering for display only; not a registration or physical-aspect-ratio validation.'
(root/'input_check_with_geometry.json').write_text(json.dumps(checks,indent=2))
bundle=root/'MRI_sub-mindo304_step1_reports.zip'
with zipfile.ZipFile(bundle,'w',compression=zipfile.ZIP_DEFLATED) as archive:
    for name in ['runtime_check.json','input_check.json','input_check_with_geometry.json','STEP1_README.txt','raw_preview.png','pilot_subjects.csv','pinned_manifest.json']:
        archive.write(root/name,arcname=name)
    for item in selected:
        sidecar=raw_root/(item['path'].removesuffix('.nii.gz')+'.json')
        archive.write(sidecar,arcname='sidecars/'+sidecar.name)
print(summary)
print('REPORT BUNDLE:',bundle,'bytes:',bundle.stat().st_size,'SHA256:',file_sha(bundle))


STEP 1 - sub-mindo304_ses-placebo
Completed: runtime inspection; verified two MRI files and matching JSON; checked full voxel readability, dimensions, TR and source metadata.
T1: 256 x 256 x 156; 1 mm header spacing; LAS.
fMRI: 64 x 64 x 43 x 240; JSON TR 2 seconds.
Runtime: 2 logical CPUs, 12.67 GiB RAM, no GPU. FreeSurfer/fMRIPrep not installed.
Review required: BOLD header x-spacing 2.0 versus sform-derived 3.4375; BOLD space/time units unknown; qform/sform mapping differs; SliceTiming unavailable.
qform/sform differences alone do not prove corruption. Establish intended coordinate mapping from source evidence before harmonizing derived inputs.
No raw headers changed; no preprocessing, motion estimation, feature extraction or training performed.
Status: file integrity PASS; geometry REVIEW REQUIRED; QC PENDING; training_ready FALSE.
Raw files remain in the temporary Colab runtime; notebook outputs are saved in Drive. Rerun download cells if runtime is recycled.
Next: resolve source 

In [7]:
# Download the report bundle only; raw MRI files are not included.
from google.colab import files
files.download(str(bundle))


In [10]:
"""Close the first-record intake with an explicit, reversible pilot header policy.

No resampling, voxel edits, MRI preprocessing, motion estimation or QC approval.
Only sub-mindo304_ses-placebo is supported; do not generalize to other records.
"""
from pathlib import Path
import hashlib
import json
import re
import urllib.request
from datetime import datetime, timezone

import nibabel as nib
import numpy as np

UPSTREAM = "0b11e8604142ebb9d335f57877238f3aad44e194"
BASE = f"https://raw.githubusercontent.com/OpenNeuroDatasets/ds007857/{UPSTREAM}/"
RID = "sub-mindo304_ses-placebo"
EXPECTED = {
    "t1": ("sub-mindo304/ses-placebo/anat/" + RID + "_T1w.nii.gz",
           "89ea431ba0e912d1b73643a4ce691dbe13ad54da394b645942e4e12dceaf73b5", 8425883),
    "bold": ("sub-mindo304/ses-placebo/func/" + RID + "_task-rest_bold.nii.gz",
             "7469f562c6fb134d53508297710e446a0d943ee4b196d2c06fb30cd8b1c2877d", 60013116),
}


def sha(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()


def public_bytes(url):
    request = urllib.request.Request(url, headers={"User-Agent": "FYP-MRI-intake-audit"})
    with urllib.request.urlopen(request, timeout=120) as response:
        return response.read()


def close_step1(root):
    root = Path(root)
    previous = json.loads((root / "input_check_with_geometry.json").read_text())
    assert previous["record_id"] == RID
    output = root / "intake_v2"
    output.mkdir(exist_ok=True)
    evidence = {
        "upstream_commit": UPSTREAM, "upstream_tag": "1.0.1",
        "protocol": "https://doi.org/10.1186/s12883-019-1294-6",
        "protocol_rest_fov_mm": [220, 220],
        "protocol_rest_tr_sec": 2.0, "protocol_rest_slices": 43,
        "protocol_rest_volumes": 240, "protocol_rest_slice_thickness_mm": 3.0,
        "policy": "pilot_sform_preserving_header_harmonization_v1",
        "inference_limit": "FOV/matrix corroborates sform; not a subject DICOM or scanner-author confirmation.",
        "sources": {}, "images": {},
    }
    # Source directories were checked for inheritance; repeat and record actual entries.
    for directory in ["", "sub-mindo304", "sub-mindo304/ses-placebo"]:
        url = "https://api.github.com/repos/OpenNeuroDatasets/ds007857/contents/" + directory + "?ref=" + UPSTREAM
        entries = json.loads(public_bytes(url))
        paths = [e["path"] for e in entries if e["type"] == "file" and e["name"].endswith(".json")]
        assert not any(p.endswith("_bold.json") for p in paths), "New inherited BOLD metadata: review policy"
        evidence["sources"][directory or "dataset_root"] = {"url": url, "json_files": paths}
    description = json.loads(public_bytes(BASE + "dataset_description.json"))
    assert "Michigan Neural Distinctiveness" in description["Name"]
    for kind, (relative, expected_sha, expected_size) in EXPECTED.items():
        raw = root / "dataset_Mi" / "rawdata" / relative
        assert raw.stat().st_size == expected_size and sha(raw) == expected_sha
        # Git-annex symlink binds the actual original image content to the SHA-256.
        annex_url = BASE + relative
        annex_target = public_bytes(annex_url).decode().strip()
        assert f"SHA256E-s{expected_size}--{expected_sha}.nii.gz" in annex_target
        json_relative = relative.removesuffix(".nii.gz") + ".json"
        original_json = public_bytes(BASE + json_relative)
        local_json = raw.with_name(raw.name.removesuffix(".nii.gz") + ".json")
        assert original_json == local_json.read_bytes(), "Local/source JSON differ"
        metadata = json.loads(original_json)
        source = nib.load(raw)
        affine, code = source.get_sform(coded=True)
        assert affine is not None and code != 0 and np.isfinite(affine).all()
        assert np.allclose(source.affine, affine)
        spacing = nib.affines.voxel_sizes(affine)
        if kind == "bold":
            assert source.shape == (64, 64, 43, 240)
            assert np.allclose(spacing, [220 / 64, 220 / 64, 3.0], atol=1e-5)
            assert metadata["RepetitionTime"] == 2.0
            assert not metadata.get("SliceTiming"), "SliceTiming newly available: review policy"
            # Do not invent timing from scanner model, number of slices or generic defaults.
            assert int(source.header["slice_code"]) == 0, "NIfTI timing code needs review"
        else:
            assert source.shape == (256, 256, 156) and np.allclose(spacing, [1, 1, 1], atol=1e-5)
        unscaled = np.asarray(source.dataobj.get_unscaled())
        destination = output / kind / raw.name
        destination.parent.mkdir(exist_ok=True)
        if not destination.exists():
            header = source.header.copy()
            header.set_xyzt_units("mm", "sec")
            header.set_zooms(tuple(spacing) + ((2.0,) if kind == "bold" else ()))
            staged = nib.Nifti1Image(unscaled, affine, header=header)
            # Preserve the full coded affine, including tiny non-orthogonality.
            # qform cannot encode shear without changing that mapping; disable it.
            staged.set_sform(affine, int(code))
            staged.set_qform(None, code=0)
            staged.header.set_xyzt_units("mm", "sec")
            staged.header.set_slope_inter(source.dataobj.slope, source.dataobj.inter)
            temporary = destination.with_name(destination.name.removesuffix(".nii.gz") + ".partial.nii.gz")
            nib.save(staged, temporary)
            temporary.rename(destination)
        check = nib.load(destination)
        assert np.array_equal(unscaled, np.asarray(check.dataobj.get_unscaled()))
        assert np.array_equal(source.get_fdata(), check.get_fdata()), "Scaled voxel values changed"
        assert check.shape == source.shape
        assert np.allclose(check.affine, source.affine, atol=1e-5)
        assert int(check.header["qform_code"]) == 0
        assert np.allclose(check.get_sform(), source.get_sform(), atol=1e-5)
        assert np.allclose(check.header.get_zooms()[:3], spacing, atol=1e-5)
        assert check.header.get_xyzt_units() == ("mm", "sec")
        assert sha(raw) == expected_sha, "Original raw image changed"
        if kind == "bold":
            assert check.header.get_zooms()[3] == 2.0
        sidecar_copy = destination.with_name(destination.name.removesuffix(".nii.gz") + ".json")
        # Preserve original JSON byte-for-byte; discrepancy is resolved by a separate explicit recipe.
        if sidecar_copy.exists():
            assert sidecar_copy.read_bytes() == original_json
        else:
            sidecar_copy.write_bytes(original_json)
        evidence["images"][kind] = {
            "raw_path": str(raw), "raw_sha256": expected_sha,
            "upstream_annex_url": annex_url, "upstream_annex_target": annex_target,
            "json_url": BASE + json_relative,
            "json_sha256": hashlib.sha256(original_json).hexdigest(),
            "staged_path": str(destination), "staged_sha256": sha(destination),
            "shape": list(source.shape), "spacing_mm": list(map(float, spacing)),
            "chosen_affine": affine.tolist(), "sform_code": int(code),
            "raw_slice_code": int(source.header["slice_code"]),
            "raw_slice_duration": float(source.header["slice_duration"]),
            "raw_header_spacing": list(map(float, source.header.get_zooms())),
            "staged_qform_code": 0,
            "coordinate_policy": "preserve exact coded sform; disable ambiguous qform instead of approximating shear",
            "original_voxels_identical": True, "world_affine_preserved": True,
        }
        if kind == "bold":
            evidence["images"][kind]["original_json_VoxelSize"] = metadata.get("VoxelSize")
            evidence["images"][kind]["VoxelSize_discrepancy"] = "Original JSON says 3,3,3; do not use that field to rescale data. Effective pilot spacing follows coded sform plus protocol FOV corroboration."
    result = {
        "record_id": RID, "closed_at_utc": datetime.now(timezone.utc).isoformat(),
        "step1_status": "complete_for_technical_pilot_with_documented_limitations",
        "file_integrity_status": "pass", "staged_geometry_checks": "pass",
        "spatial_policy": evidence["policy"],
        "slice_timing_correction": "skip_missing_source_metadata_and_unknown_nifti_slice_code",
        "fieldmap_based_sdc": "unavailable_no_fieldmap_acquisition_in_checked_session",
        "tr_sec": 2.0, "resampling_performed": False, "voxel_values_changed": False,
        "scientific_preprocessing_started": False, "preprocessed_handoff_ready": False,
        "qc_status": "pending", "training_ready": False,
        "identity_status": previous["identity_status"], "diagnosis_status": previous["diagnosis_status"],
        "runtime": "2 logical CPUs / 12.67 GiB RAM / CPU only; no full FreeSurfer/fMRIPrep environment verified",
        "remaining_stage2_requirements": [
            "Install and verify preprocessing software and persistent checkpoint storage",
            "Confirm BrainIAC template/input recipe with feature owner",
            "Lock fMRI denoising/QC parameters and exact Brain-JEPA atlas/order before extraction",
            "Perform actual preprocessing and subsequent manual QC; do not mark ready now",
        ],
        "evidence": evidence,
    }
    (output / "step1_closeout.json").write_text(json.dumps(result, indent=2))
    return result


closeout=close_step1(Path("/content/FYP_MRI_LiYang"))
print(json.dumps({k:v for k,v in closeout.items() if k!="evidence"},indent=2))
print("STAGED IMAGE CHECKS:",json.dumps({k:{field:v[field] for field in ["shape","spacing_mm","original_voxels_identical","world_affine_preserved","staged_sha256"]} for k,v in closeout["evidence"]["images"].items()},indent=2))


{
  "record_id": "sub-mindo304_ses-placebo",
  "closed_at_utc": "2026-11-09T15:12:31.026791+00:00",
  "step1_status": "complete_for_technical_pilot_with_documented_limitations",
  "file_integrity_status": "pass",
  "staged_geometry_checks": "pass",
  "spatial_policy": "pilot_sform_preserving_header_harmonization_v1",
  "slice_timing_correction": "skip_missing_source_metadata_and_unknown_nifti_slice_code",
  "fieldmap_based_sdc": "unavailable_no_fieldmap_acquisition_in_checked_session",
  "tr_sec": 2.0,
  "resampling_performed": false,
  "voxel_values_changed": false,
  "scientific_preprocessing_started": false,
  "preprocessed_handoff_ready": false,
  "qc_status": "pending",
  "training_ready": false,
  "identity_status": "unconfirmed",
  "diagnosis_status": "timing_unverified",
  "runtime": "2 logical CPUs / 12.67 GiB RAM / CPU only; no full FreeSurfer/fMRIPrep environment verified",
  "remaining_stage2_requirements": [
    "Install and verify preprocessing software and persistent che

In [11]:
# Final Step 1 record. This supersedes the earlier 'review required' intake snapshot.
from google.colab import files
import zipfile
close_path=Path('/content/FYP_MRI_LiYang/intake_v2/step1_closeout.json')
final_bundle=Path('/content/FYP_MRI_LiYang/MRI_sub-mindo304_step1_reports_final.zip')
with zipfile.ZipFile(final_bundle,'w',compression=zipfile.ZIP_DEFLATED) as out:
    with zipfile.ZipFile('/content/FYP_MRI_LiYang/MRI_sub-mindo304_step1_reports.zip') as old:
        for name in old.namelist():
            out.writestr('initial_snapshot/'+name,old.read(name))
    out.write(close_path,arcname='step1_closeout.json')
    out.writestr('FINAL_README.txt','First-stage intake closed for a technical pilot with documented limitations. Read step1_closeout.json for the final decision; initial_snapshot records earlier findings. Raw originals unchanged. Staged copies preserve all voxel values and the existing coded sform; qform disabled, spatial units mm, temporal units seconds, BOLD TR=2 seconds. Missing SliceTiming: skip STC; no fieldmap-based SDC in this session. This is not completed preprocessing or patient QC. Source FOV/matrix corroborates the pilot sform choice; no per-subject DICOM confirmation. Model-specific template, denoising/QC and atlas recipes remain stage-2 prerequisites. Raw/staged MRI files are not in this report bundle; they remain in the temporary Colab runtime and can be reconstructed from pinned sources and code. No training readiness granted.')
print('STEP 1 CLOSED FOR TECHNICAL PILOT')
print('Original source hashes verified: T1 + BOLD + matching JSON')
print('Staged spacing: T1 1x1x1 mm; BOLD 3.4375x3.4375x3 mm; TR 2 seconds')
print('Voxel values unchanged; chosen world affine preserved; no resampling')
print('Missing SliceTiming: skip STC; no fieldmap-based SDC')
print('Preprocessing not started; QC pending; training_ready False')
print('Final report SHA256:',sha(final_bundle))
files.download(str(final_bundle))
files.download(str(close_path))


STEP 1 CLOSED FOR TECHNICAL PILOT
Original source hashes verified: T1 + BOLD + matching JSON
Staged spacing: T1 1x1x1 mm; BOLD 3.4375x3.4375x3 mm; TR 2 seconds
Voxel values unchanged; chosen world affine preserved; no resampling
Missing SliceTiming: skip STC; no fieldmap-based SDC
Preprocessing not started; QC pending; training_ready False
Final report SHA256: 901aca7331c13778619675a86669c810407ff759892dbdb121b5c5812e887062
